In [3]:
import json
import os
from time import time

from dotenv import load_dotenv
from azure.cosmos import CosmosClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
import requests
from pathlib import Path
from openai import AzureOpenAI
from Backend.models import Product, ProductList

import json
from pydantic import BaseModel, ValidationError
from typing import Type, TypeVar, List
from azure.cosmos import CosmosClient, DatabaseProxy, ContainerProxy, PartitionKey
import time
#from tenacity import retry, wait_random_exponential, stop_after_attempt

loaded = load_dotenv()
if not loaded:
    raise RuntimeError(".env file could not be loaded.")

key_vault_url = os.environ["KEY_VAULT_URL"]
secret_name = os.environ["COSMOS_DB_CONNECTION_STRING_SECRET_NAME"]

credential = DefaultAzureCredential()
secret_client = SecretClient(vault_url=key_vault_url, credential=credential)
connection_string = secret_client.get_secret(secret_name).value

if connection_string is None:
    raise RuntimeError(f"Key Vault secret '{secret_name}' has no value.")

client = CosmosClient.from_connection_string(connection_string)

DatabaseProxy = client.create_database_if_not_exists("test_db_5")

ContainerProxy = DatabaseProxy.create_container_if_not_exists(
    id="product",
    partition_key={"paths": ["/categoryId"], "kind": "Hash"}
)

# Create a document
# ContainerProxy.upsert_item({"id": "1", "categoryId": "electronics"})

# items = ContainerProxy.query_items(query="SELECT * FROM product", enable_cross_partition_query=True)

# Add product data to database using batch operations
# Get cosmic works product data from github
# product_raw_data = "https://cosmosdbcosmicworks.blob.core.windows.net/cosmic-works-small/product.json"
# product_data = ProductList(items=[Product(**data) for data in requests.get(product_raw_data).json()])
# for prod in product_data.items:
#     print(prod)
#     ContainerProxy.upsert_item(prod.model_dump(by_alias=True))

# product_raw_data = "https://cosmosdbcosmicworks.blob.core.windows.net/cosmic-works-small/product.json"
# product_data = ProductList(items=[Product(**data) for data in requests.get(product_raw_data).json()])
# for prod in product_data.items:
#     ContainerProxy.upsert_item(prod.model_dump(by_alias=True))

# product_file = Path("products.json")
# with product_file.open(encoding="utf-8") as file:
#     product_data = ProductList(
#         items=[Product(**data) for data in json.load(file)]
#     )

# for prod in product_data.items:
#     ContainerProxy.upsert_item(prod.model_dump(by_alias=True))

aoai_endpoint = os.environ["AOAI_ENDPOINT"]
api_version = os.environ["AOAI_API_VERSION"]
aoai_key = os.environ["AOAI_KEY"]

# Instantiate an AzureOpenAI client
ai_client = AzureOpenAI(
    azure_endpoint = aoai_endpoint,
    api_version = api_version,
    api_key = aoai_key
    )

# Generate embeddings from a text string
embeddings_deployment_name = os.environ["EMBEDDINGS_DEPLOYMENT_NAME"]

def generate_embeddings(text: str):
    '''
    Generate embeddings from string of text using the deployed Azure OpenAI API embeddings model.
    This will be used to vectorize document data and incoming user messages for a similarity search with
    the vector index.
    '''
    response = ai_client.embeddings.create(input=text, model=embeddings_deployment_name)
    embeddings = response.data[0].embedding
    time.sleep(0.5) # rest period to avoid rate limiting on AOAI
    return embeddings

# Create the vector embedding policy
vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/contentVector",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": 1536
        }
    ]
}

# Create the indexing policy
indexing_policy = {
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [
        {
            "path": "/*"
        }
    ],
    "excludedPaths": [
        {
            "path": "/\"_etag\"/?"
        },
        {
            "path": "/contentVector/*"
        }
    ],
    "vectorIndexes": [
        {
            "path": "/contentVector",
            "type": "diskANN"
        }
    ]
}

product_v_container = DatabaseProxy.create_container_if_not_exists(
    id="product_v",
    partition_key=PartitionKey(path="/categoryId"),
    indexing_policy=indexing_policy,
    vector_embedding_policy=vector_embedding_policy
)

### Performing a vector search query
def vector_search(
        container: ContainerProxy, 
        prompt: str,         
        vector_field_name:str="contentVector", 
        num_results:int=5):
    query_embedding = generate_embeddings(prompt)    
    items = container.query_items(
        query=f"""SELECT TOP @num_results itm.id, VectorDistance(itm.{vector_field_name}, @embedding) AS SimilarityScore 
                FROM itm
                ORDER BY VectorDistance(itm.{vector_field_name}, @embedding)
                """,
        parameters = [
            { "name": "@num_results", "value": num_results },
            { "name": "@embedding", "value": query_embedding }            
        ],
        enable_cross_partition_query=True
        )
    return items

### Using vector search results with an LLM in a RAG scenario
def rag_with_vector_search(
        container: ContainerProxy, 
        prompt: str,         
        vector_field_name:str="contentVector", 
        num_results:int=5):
    """
    Use the RAG model to generate a prompt using vector search results based on the
    incoming question.
    """
    # perform the vector search and build product list
    results = vector_search(container, prompt, vector_field_name, num_results)
    product_list = ""
    for result in results:
        # retrieve the product details
        product = query_item_by_id(container, result["id"], Product)               
        # remove the contentVector field from the product details, this isn't needed for the context
        product.content_vector = None        
        product_list += json.dumps(product, indent=4, default=str) + "\n\n"

    # generate prompt for the LLM using vector search results
    formatted_prompt = system_prompt + product_list

    # prepare the LLM request
    messages = [
        {"role": "system", "content": formatted_prompt},
        {"role": "user", "content": prompt}
    ]

    completion = ai_client.chat.completions.create(messages=messages, model=deployment_name)
    return completion.choices[0].message.content

### Instantiating a vector store reference

from azure.cosmos import CosmosClient
from langchain_azure_ai.vectorstores.azure_cosmos_db_no_sql import (
    AzureCosmosDBNoSqlVectorSearch,
)

cosmos_client = CosmosClient.from_connection_string(CONNECTION_STRING)

vector_store = AzureCosmosDBNoSqlVectorSearch(
    cosmos_client = cosmos_client,
    database_name = "cosmic_works",
    container_name = "products",
    embedding = embedding_model,
    vector_embedding_policy = vector_embedding_policy,
    indexing_policy = indexing_policy
)

# Create a retriever from the vector store
retriever = vector_store.as_retriever()

# Create the prompt template from the system_prompt text
llm_prompt = PromptTemplate.from_template(system_prompt)

rag_chain = (
    # populate the tokens/placeholders in the llm_prompt
    # products takes the results of the vector store and formats the documents
    # question is a passthrough that takes the incoming question
    { "products": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    # pass the populated prompt to the language model
    | llm
    # return the string ouptut from the language model
    | StrOutputParser()
)

# create a chain on the retriever to format the documents as JSON
products_retriever_chain = products_retriever | format_docs

tools = [
    Tool(
        name = "vector_search_products",
        func = products_retriever_chain.invoke,
        description = "Searches Cosmic Works product information for similar products based on the question. Returns the product information in JSON format."
    )
]

container = CosmosClient(CONNECTION_STRING).get_database_client("cosmic_works").get_container_client("products")

def get_product_by_id(product_id: str) -> str:
    """
    Retrieves a product by its ID.
    """
    doc = container.read_item(item=product_id, partition_key=product_id)
    if "contentVector" in doc:
        del doc["contentVector"]
    return json.dumps(doc)

def get_product_by_sku(sku: str) -> str:
    """
    Retrieves a product by its sku.
    """
    query = f"SELECT * FROM c WHERE c.sku = '{sku}'"
    results = list(container.query_items(query=query, enable_cross_partition_query=True))
    if results:
        doc = results[0]
        if "contentVector" in doc:
            del doc["contentVector"]
        return json.dumps(doc, default=str)
    return "{}"

from langchain.tools import StructuredTool

tools.extend([
    StructuredTool.from_function(get_product_by_id),
    StructuredTool.from_function(get_product_by_sku)
    #StructuredTool.from_function(get_sales_by_id)
])

### Creating an agent armed with tools for vector search and Python functions calling

system_message = SystemMessage(
    content = """
        You are a helpful, fun and friendly sales assistant for Cosmic Works, a bicycle and bicycle accessories store.

        Your name is Cosmo.

        You are designed to answer questions about the products that Cosmic Works sells, the customers that buy them, and the sales orders that are placed by customers.

        If you don't know the answer to a question, respond with "I don't know."

        Only answer questions related to Cosmic Works products, customers, and sales orders.

        If a question is not related to Cosmic Works products, customers, or sales orders,
        respond with "I only answer questions about Cosmic Works"
    """
)
agent_executor = create_conversational_retrieval_agent(llm, tools, system_message = system_message, verbose=True)

C:\Users\apetrut\AppData\Local\Temp\ipykernel_15688\2190210640.py:195: DeprecationWarning: Importing AzureCosmosDBNoSqlVectorSearch from 'langchain_azure_ai.vectorstores.azure_cosmos_db_no_sql' is deprecated. Use 'from langchain_azure_cosmosdb import AzureCosmosDBNoSqlVectorSearch' instead.
  from langchain_azure_ai.vectorstores.azure_cosmos_db_no_sql import (


NameError: name 'CONNECTION_STRING' is not defined